Set up and import the MD17 data container

In [2]:
from __future__ import annotations

import io
import re
import zipfile
from dataclasses import dataclass
from pathlib import Path
from typing import Optional

import numpy as np
import torch

REQUIRED_KEYS = ("z", "R", "E", "F")

# Minimal symbol -> atomic number table covering every element that shows up
# in the current bronze roster (glossary.md §3: C, H, N, O only across
# MD17/MD22). Extend if a new molecule introduces a new element.
_PERIODIC_TABLE = {"H": 1, "C": 6, "N": 7, "O": 8, "F": 9, "S": 16, "Cl": 17}


@dataclass
class MD17Sample:
    """One molecule+theory's worth of trajectory data, already validated."""

    molecule: str
    theory: str
    z: np.ndarray  # [n_atoms]
    R: np.ndarray  # [n_configs, n_atoms, 3]
    E: np.ndarray  # [n_configs]
    F: np.ndarray  # [n_configs, n_atoms, 3]

    def __post_init__(self) -> None:
        self.z = np.asarray(self.z, dtype=np.int64).reshape(-1)
        self.R = np.asarray(self.R, dtype=np.float32)
        self.F = np.asarray(self.F, dtype=np.float32)
        self.E = np.asarray(self.E, dtype=np.float32).reshape(-1)

        if self.R.ndim == 2:  # single configuration provided unbatched
            self.R = self.R[None, ...]
        if self.F.ndim == 2:
            self.F = self.F[None, ...]

        n_atoms = self.z.shape[0]
        if self.R.shape[1:] != (n_atoms, 3):
            raise ValueError(
                f"{self.molecule}: R shape {self.R.shape} inconsistent with {n_atoms} atoms"
            )
        if self.F.shape != self.R.shape:
            raise ValueError(
                f"{self.molecule}: F shape {self.F.shape} != R shape {self.R.shape}"
            )
        if self.E.shape[0] != self.R.shape[0]:
            raise ValueError(
                f"{self.molecule}: E has {self.E.shape[0]} entries, "
                f"R has {self.R.shape[0]} configs"
            )

    @property
    def n_atoms(self) -> int:
        return int(self.z.shape[0])

    @property
    def n_configs(self) -> int:
        return int(self.R.shape[0])

In [3]:
def _infer_theory(filename: str) -> str:
    name = filename.lower()
    if "ccsd_t" in name or "ccsd-t" in name:
        return "ccsd_t"
    if "ccsd" in name:
        return "ccsd"
    return "dft"


def _infer_molecule(filename: str) -> str:
    stem = Path(filename).stem
    stem = re.sub(r"^md1[78]_", "", stem)
    stem = re.sub(r"[_-]?(dft|ccsd_t|ccsd-t|ccsd|train|test)$", "", stem, flags=re.IGNORECASE)
    return stem


def load_npz(
    path: str | Path, molecule: Optional[str] = None, theory: Optional[str] = None
) -> MD17Sample:
    """Load one raw sGDML-format .npz file (MD17 or MD22)."""
    path = Path(path)
    data = np.load(path, allow_pickle=True)

    missing = [k for k in REQUIRED_KEYS if k not in data]
    if missing:
        raise ValueError(f"{path.name}: missing required keys {missing}, has {list(data.keys())}")

    inferred_molecule = str(data["name"]) if "name" in data.files else _infer_molecule(path.name)

    return MD17Sample(
        molecule=molecule or inferred_molecule,
        theory=theory or _infer_theory(path.name),
        z=data["z"],
        R=data["R"],
        E=data["E"],
        F=data["F"],
    )


def _split_xyz_frames(text: str) -> list[str]:
    """Split a multi-frame concatenated .xyz file into per-frame text blocks."""
    lines = text.strip().splitlines()
    frames = []
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        n_atoms = int(lines[i].strip())
        frames.append("\n".join(lines[i : i + 2 + n_atoms]))
        i += 2 + n_atoms
    return frames


def _parse_xyz_frame(text: str) -> tuple[np.ndarray, float, np.ndarray, np.ndarray]:
    """Parse one frame of sGDML's extended-xyz-with-forces format:

        <n_atoms>
        <comment line, contains the energy>
        <element> <x> <y> <z> <fx> <fy> <fz>
        ... (n_atoms lines"""
    lines = [l for l in text.strip().splitlines() if l.strip()]
    n_atoms = int(lines[0].strip())
    comment = lines[1]

    energy_match = re.search(r"[-+]?\d*\.\d+(?:[eE][-+]?\d+)?", comment)
    energy = float(energy_match.group()) if energy_match else float("nan")

    symbols, coords, forces = [], [], []
    for line in lines[2 : 2 + n_atoms]:
        parts = line.split()
        symbols.append(parts[0])
        coords.append([float(x) for x in parts[1:4]])
        forces.append([float(x) for x in parts[4:7]])

    z = np.array([_PERIODIC_TABLE.get(s, 0) for s in symbols], dtype=np.int64)
    return z, energy, np.array(coords, dtype=np.float32), np.array(forces, dtype=np.float32)

In [4]:
def load_split_zip(
    path: str | Path, molecule: Optional[str] = None, theory: Optional[str] = None
) -> dict[str, MD17Sample]:
    """Load a literature-standard train/test split shipped as a .zip.

    Handles both shapes seen in the current bronze roster (glossary.md §3):
      - zip containing '*-train.npz' / '*-test.npz' (already canonical)
      - zip containing '*-train.xyz' / '*-test.xyz' (parsed frame-by-frame)

    Returns {"train": MD17Sample, "test": MD17Sample}.
    """
    path = Path(path)
    molecule = molecule or _infer_molecule(path.name)
    theory = theory or _infer_theory(path.name)

    result: dict[str, MD17Sample] = {}
    with zipfile.ZipFile(path) as zf:
        names = zf.namelist()
        npz_members = [n for n in names if n.lower().endswith(".npz") and not _is_junk_member(n)]
        xyz_members = [n for n in names if n.lower().endswith(".xyz") and not _is_junk_member(n)]

        if npz_members:
            for member in npz_members:
                split = _split_name(member)
                if split is None:
                    continue
                with zf.open(member) as f:
                    buf = io.BytesIO(f.read())
                    data = np.load(buf, allow_pickle=True)
                    result[split] = MD17Sample(
                        molecule=molecule, theory=theory,
                        z=data["z"], R=data["R"], E=data["E"], F=data["F"],
                    )
        elif xyz_members:
            for member in xyz_members:
                split = _split_name(member)
                if split is None:
                    continue
                with zf.open(member) as f:
                    text = f.read().decode("utf-8")
                z = None
                Rs, Es, Fs = [], [], []
                for frame_text in _split_xyz_frames(text):
                    fz, e, r, force = _parse_xyz_frame(frame_text)
                    z = fz if z is None else z
                    Rs.append(r)
                    Es.append(e)
                    Fs.append(force)
                result[split] = MD17Sample(
                    molecule=molecule, theory=theory,
                    z=z, R=np.stack(Rs), E=np.array(Es), F=np.stack(Fs),
                )
        else:
            raise ValueError(f"{path.name}: no .npz or .xyz members found ({names})")

    if "train" not in result or "test" not in result:
        raise ValueError(f"{path.name}: expected both train and test splits, found {list(result.keys())}")

    return result


def _is_junk_member(member: str) -> bool:
    base = Path(member).name
    return member.startswith("__MACOSX/") or base.startswith("._") or member.endswith("/")


def _split_name(member: str) -> Optional[str]:
    lower = member.lower()
    if "train" in lower:
        return "train"
    if "test" in lower:
        return "test"
    return None

In [5]:
def build_graph(R_frame: np.ndarray, cutoff: float = 5.0) -> tuple[torch.Tensor, torch.Tensor]:
    """Build a radius-cutoff neighbor graph for one atomic configuration.

    Nodes = atoms. Edges connect atom pairs within `cutoff` Angstrom.
    Edge feature = interatomic distance (glossary.md §5).

    Args:
        R_frame: float[n_atoms, 3] coordinates for a single configuration.
        cutoff: neighbor cutoff radius in Angstrom (default 5.0, glossary.md §5).

    Returns:
        edge_index: long[2, n_edges]
        edge_attr:  float[n_edges, 1] interatomic distances
    """
    pos = torch.as_tensor(R_frame, dtype=torch.float32)
    dist = torch.cdist(pos, pos)  # [n_atoms, n_atoms]
    n_atoms = pos.shape[0]
    mask = (dist <= cutoff) & ~torch.eye(n_atoms, dtype=torch.bool)
    row, col = mask.nonzero(as_tuple=True)
    edge_index = torch.stack([row, col], dim=0)
    edge_attr = dist[row, col].unsqueeze(-1)
    return edge_index, edge_attr


def edge_distances(pos: torch.Tensor, edge_index: torch.Tensor) -> torch.Tensor:
    """Recompute edge distances from a *live* pos tensor, preserving autograd.

    Use this at training/inference time when forces are needed: take the
    static edge_index from build_graph() (topology doesn't need to be
    differentiable), but recompute the actual distance values from a pos
    tensor that has requires_grad=True, so that
    torch.autograd.grad(energy, pos) produces correct, non-zero forces.

    Args:
        pos: float[n_atoms, 3], must be the tensor forces will be
             differentiated with respect to (requires_grad=True).
        edge_index: long[2, n_edges], from build_graph().

    Returns:
        edge_attr: float[n_edges, 1], differentiably connected to pos.
    """
    row, col = edge_index
    return (pos[row] - pos[col]).norm(dim=-1, keepdim=True)